# Notebook 06: Real Guardrail Classifier + Production Capstone

`[REAL]` Companion to Modules 07-09. A real, local Hugging Face toxicity classifier run on the RTX 4060, evaluated for real precision/recall/F1 across a real swept threshold range with a principled, stated threshold-selection method. Real measured sequential-vs-parallel guardrail latency using two genuinely independent checks. A two-part capstone: (A) a real per-request pipeline trace with root-cause localization (Module 07's own function, reused verbatim), and (B) a separate, explicitly `[SIMULATION]`-labeled aggregate evaluation-set-versioning comparison (Module 09's own functions, reused verbatim) -- kept as two distinct experiments answering two different real questions, per the signed-off plan.

In [1]:
import os
import re
import time
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
GEN_MODEL = "gpt-4o-mini"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"OpenAI client ready. Generation model: {GEN_MODEL}. Local classifier device: {DEVICE}")

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAI client ready. Generation model: gpt-4o-mini. Local classifier device: cuda


## 1. Real Local Toxicity Classifier + Real Labeled Test Set

`[REAL]` A real, small pretrained Hugging Face toxicity classifier (`unitary/toxic-bert`), loaded and run locally on the RTX 4060 -- more representative of real production guardrail deployment than a remote API call.

**Labeling protocol (stated before scoring, mirroring Notebook 01's own correctness protocol):** an item is labeled toxic only if it contains a direct hostile insult or threat aimed at a person; it is labeled non-toxic if it is neutral, informational, or positive in content. This is a real, explicit, reproducible rule, not a subjective per-item judgment.

In [2]:
TOXICITY_MODEL_NAME = "unitary/toxic-bert"
tox_tokenizer = AutoTokenizer.from_pretrained(TOXICITY_MODEL_NAME)
tox_model = AutoModelForSequenceClassification.from_pretrained(TOXICITY_MODEL_NAME).to(DEVICE)
tox_model.eval()
TOXIC_LABEL_INDEX = [i for i, lbl in tox_model.config.id2label.items() if lbl == "toxic"][0]
print(f"Real classifier loaded: {TOXICITY_MODEL_NAME} on {DEVICE}. "
      f"Real label set: {list(tox_model.config.id2label.values())}")

def toxicity_score(text):
    inputs = tox_tokenizer(text, return_tensors="pt", truncation=True).to(DEVICE)
    with torch.no_grad():
        logits = tox_model(**inputs).logits
    probs = torch.sigmoid(logits)[0]
    return probs[TOXIC_LABEL_INDEX].item()

TEST_SET = [
    ("You are a complete idiot and everyone hates you.", 1),
    ("Get lost, nobody wants you here, you worthless piece of trash.", 1),
    ("I will find you and make you regret this, you disgusting coward.", 1),
    ("Shut up, you stupid moron, nobody cares what you think.", 1),
    ("You're pathetic and everyone can see how useless you are.", 1),
    ("Go away, you filthy idiot, and never come back.", 1),
    ("You're an absolute waste of space and everyone is sick of you.", 1),
    ("Nobody asked for your stupid opinion, keep it to yourself, loser.", 1),
    ("The weather today is sunny with a light breeze.", 0),
    ("Thank you so much for your help, I really appreciate it.", 0),
    ("The quarterly report is due on Friday at 5pm.", 0),
    ("Could you please pass the salt?", 0),
    ("I think this restaurant has great pasta.", 0),
    ("The train departs from platform 4 at 9:15am.", 0),
    ("Congratulations on your promotion, well deserved!", 0),
    ("I disagree with this approach, but I respect your reasoning.", 0),
]

scored_test_set = [(text, label, toxicity_score(text)) for text, label in TEST_SET]
for text, label, score in scored_test_set:
    print(f"label={label}  real_score={score:.4f}  {text!r}")

D:\Study\Prep\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aryan\.cache\huggingface\hub\models--unitary--toxic-bert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7833.92it/s]

Real classifier loaded: unitary/toxic-bert on cuda. Real label set: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


label=1  real_score=0.9907  'You are a complete idiot and everyone hates you.'
label=1  real_score=0.9869  'Get lost, nobody wants you here, you worthless piece of trash.'
label=1  real_score=0.9833  'I will find you and make you regret this, you disgusting coward.'
label=1  real_score=0.9955  'Shut up, you stupid moron, nobody cares what you think.'
label=1  real_score=0.9796  "You're pathetic and everyone can see how useless you are."
label=1  real_score=0.9957  'Go away, you filthy idiot, and never come back.'
label=1  real_score=0.9692  "You're an absolute waste of space and everyone is sick of you."
label=1  real_score=0.9708  'Nobody asked for your stupid opinion, keep it to yourself, loser.'
label=0  real_score=0.0006  'The weather today is sunny with a light breeze.'
label=0  real_score=0.0005  'Thank you so much for your help, I really appreciate it.'
label=0  real_score=0.0006  'The quarterly report is due on Friday at 5pm.'
label=0  real_score=0.0015  'Could you please pass 

`[REAL]` The real local classifier scored the 8 real toxic items at `0.9692`-`0.9957` and the 8 real non-toxic items at `0.0005`-`0.0015` -- a large, clean real separation with no scores anywhere near the decision boundary. This real 16-item test set (deliberately unambiguous, per the stated labeling protocol) turned out to be easy for a real, pretrained toxicity classifier to separate perfectly.

## 2. Real Threshold Sweep + Principled Threshold Selection

`[REAL]` Real precision/recall/F1 (Module 08's own `precision_recall_f1`, reused verbatim) computed across a real swept range of decision thresholds on Section 1's real scored test set. **Threshold selection method, stated explicitly:** the threshold used in later sections is the one that maximizes real F1 -- a principled, stated criterion, not an arbitrary choice.

In [3]:
def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sweep_results = []
for t in THRESHOLDS:
    tp = sum(1 for _, label, score in scored_test_set if label == 1 and score >= t)
    fp = sum(1 for _, label, score in scored_test_set if label == 0 and score >= t)
    fn = sum(1 for _, label, score in scored_test_set if label == 1 and score < t)
    metrics = precision_recall_f1(tp, fp, fn)
    sweep_results.append({"threshold": t, "tp": tp, "fp": fp, "fn": fn, **metrics})
    print(f"t={t:.1f}  tp={tp} fp={fp} fn={fn}  "
          f"precision={metrics['precision']:.4f} recall={metrics['recall']:.4f} f1={metrics['f1']:.4f}")

CHOSEN_THRESHOLD = max(sweep_results, key=lambda r: r["f1"])["threshold"]
print(f"\nReal F1-maximizing threshold selected: {CHOSEN_THRESHOLD}")

t=0.1  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000
t=0.2  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000
t=0.3  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000
t=0.4  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000
t=0.5  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000
t=0.6  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000
t=0.7  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000
t=0.8  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000
t=0.9  tp=8 fp=0 fn=0  precision=1.0000 recall=1.0000 f1=1.0000

Real F1-maximizing threshold selected: 0.1


`[REAL]` Because Section 1's real scores were so cleanly separated, real precision/recall/F1 came out at a perfect `1.0000`/`1.0000`/`1.0000` at **every** swept threshold from `0.1` to `0.9` -- there is no real trade-off curve to observe on this particular test set. `CHOSEN_THRESHOLD` was mechanically selected as `0.1`, but this is an honest artifact of the sweep order and Python's `max()` tie-breaking (it returns the first item achieving the maximum, and every threshold tied at F1=1.0), not evidence that `0.1` is meaningfully better than `0.5` or `0.8` here. This is a real, worth-naming limitation of this notebook's own test set: it is too cleanly separable to exercise a genuine precision/recall trade-off the way Module 08's own two-threshold constructed example does -- a real production test set would need borderline, ambiguous examples near the real decision boundary to make threshold selection actually consequential.

## 3. Real Sequential vs. Parallel Guardrail Latency (Genuinely Independent Checks)

`[REAL]` Two genuinely independent checks with no data dependency between them: the real local toxicity classifier (GPU) and a real, deterministic regex-based PII detector (CPU, no model weights, no shared computation) -- satisfying the plan's independence requirement, unlike two calls to the same classifier. Real wall-clock latency measured for both a sequential loop and a `ThreadPoolExecutor`-parallel run across the same real inputs.

In [4]:
PII_PATTERNS = {
    "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    "phone": re.compile(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b"),
    "ssn": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
}

def detect_pii(text):
    for kind, pattern in PII_PATTERNS.items():
        if pattern.search(text):
            return True, kind
    return False, None

LATENCY_TEST_TEXTS = [text for text, _ in TEST_SET[:8]] + [
    "Please call me back at 555-234-7788 when you get a chance.",
    "You can reach the sales desk at sales@example.com anytime.",
]

def run_sequential(text):
    t0 = time.perf_counter()
    _ = toxicity_score(text)
    _ = detect_pii(text)
    return time.perf_counter() - t0

def run_parallel(text):
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=2) as pool:
        fut_tox = pool.submit(toxicity_score, text)
        fut_pii = pool.submit(detect_pii, text)
        fut_tox.result()
        fut_pii.result()
    return time.perf_counter() - t0

sequential_times = [run_sequential(t) for t in LATENCY_TEST_TEXTS]
parallel_times = [run_parallel(t) for t in LATENCY_TEST_TEXTS]

mean_sequential_ms = sum(sequential_times) / len(sequential_times) * 1000
mean_parallel_ms = sum(parallel_times) / len(parallel_times) * 1000
savings_pct = (mean_sequential_ms - mean_parallel_ms) / mean_sequential_ms * 100

print(f"Real mean sequential latency: {mean_sequential_ms:.2f}ms")
print(f"Real mean parallel latency:   {mean_parallel_ms:.2f}ms")
print(f"Real savings: {savings_pct:.1f}%")

Real mean sequential latency: 22.90ms
Real mean parallel latency:   27.89ms
Real savings: -21.8%


`[REAL]` The real measured result was a **negative** savings of `-21.8%`: mean sequential latency `22.90ms` vs. mean parallel latency `27.89ms` -- parallel was real slower than sequential on this notebook's own hardware, not faster. This is an honest, real counter-example to the intuition that independent checks always benefit from parallelization: the real regex-based PII check is sub-millisecond, so there is almost no real latency to hide behind the GPU classifier call, while `run_parallel` pays a real, measurable cost every call for creating a fresh `ThreadPoolExecutor`, submitting futures, and thread scheduling -- overhead Module 08's own `guardrail_latency` formula does not model, since it explicitly assumes zero orchestration overhead (stated in that module as an explicit independence assumption). This real result validates that stated assumption's importance directly: real orchestration overhead is not free, and here it exceeds the real benefit of overlapping a fast check with a slow one. Parallelizing guardrail checks is only a real net win when the checks are comparably expensive and the orchestration overhead is amortized across many concurrent requests (e.g., a persistent thread/process pool), not re-created per single item as this notebook's own minimal implementation does.

## 4. Capstone Part A: Real Per-Request Trace + Root-Cause Localization

`[REAL]` `Span`, `total_latency_ms`, and `localize_root_cause` are reused verbatim from Module 07. A real, small 3-step pipeline (retrieve -> generate -> guardrail check, reusing this notebook's own real Section 1-3 classifier/detector functions) runs for 3 real requests against a real, tiny fixed document set, with real per-span timing and status logged.

In [5]:
@dataclass
class Span:
    name: str
    latency_ms: float
    status: str
    detail: str

def total_latency_ms(spans):
    return sum(s.latency_ms for s in spans)

def localize_root_cause(spans):
    for span in spans:
        if span.status != "ok":
            return span
    return None

DOCS = [
    {"id": "returns", "keywords": ["return", "refund", "electronics"],
     "text": "Our return policy allows electronics to be returned within 30 days of purchase for a full refund, provided the original packaging is included."},
    {"id": "support", "keywords": ["contact", "support", "help"],
     "text": "For further assistance, contact our support team at 555-201-4488 or email support@example.com."},
    {"id": "hours", "keywords": ["hours", "open", "store"],
     "text": "Our stores are open Monday through Saturday, 9am to 8pm, and Sunday 10am to 6pm."},
]

def retrieve(query):
    query_words = set(re.findall(r"[a-z']+", query.lower()))
    best_doc, best_overlap = None, 0
    for doc in DOCS:
        overlap = len(query_words & set(doc["keywords"]))
        if overlap > best_overlap:
            best_doc, best_overlap = doc, overlap
    return best_doc

def generate(query, context_text):
    prompt = (
        f"Context: {context_text or '(no relevant context found)'}\n\n"
        f"Question: {query}\n\n"
        "Answer using ONLY the context above. Be specific and include any relevant contact details. "
        "If the context does not contain the answer, say you don't have that information."
    )
    resp = client.chat.completions.create(
        model=GEN_MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=80,
    )
    return resp.choices[0].message.content.strip()

def guardrail_check(answer):
    score = toxicity_score(answer)
    has_pii, pii_kind = detect_pii(answer)
    if score >= CHOSEN_THRESHOLD:
        return "flagged", f"toxicity score {score:.3f} >= threshold {CHOSEN_THRESHOLD}"
    if has_pii:
        return "flagged", f"PII detected: {pii_kind}"
    return "ok", f"toxicity score {score:.3f}, no PII detected"

def run_traced_request(query):
    spans = []

    t0 = time.perf_counter()
    doc = retrieve(query)
    retrieve_ms = (time.perf_counter() - t0) * 1000
    if doc is None:
        spans.append(Span("retrieve", retrieve_ms, "error", "no relevant document found"))
        context_text = ""
    else:
        spans.append(Span("retrieve", retrieve_ms, "ok", f"matched doc: {doc['id']}"))
        context_text = doc["text"]

    t0 = time.perf_counter()
    answer = generate(query, context_text)
    generate_ms = (time.perf_counter() - t0) * 1000
    spans.append(Span("generate", generate_ms, "ok", answer[:80]))

    t0 = time.perf_counter()
    guard_status, guard_detail = guardrail_check(answer)
    guard_ms = (time.perf_counter() - t0) * 1000
    spans.append(Span("guardrail_check", guard_ms, guard_status, guard_detail))

    return spans, answer

REQUESTS = [
    "What is your return policy for electronics?",
    "What is the CEO's home address?",
    "How do I contact your support team?",
]

traces = []
for query in REQUESTS:
    spans, answer = run_traced_request(query)
    root_cause = localize_root_cause(spans)
    traces.append({"query": query, "spans": spans, "answer": answer, "root_cause": root_cause})
    print(f"Query: {query!r}")
    print(f"  Real answer: {answer!r}")
    for s in spans:
        print(f"  Span: {s.name:<18} status={s.status:<7} latency={s.latency_ms:.1f}ms  detail={s.detail}")
    print(f"  Real total latency: {total_latency_ms(spans):.1f}ms")
    print(f"  Real localized root cause: {root_cause.name if root_cause else 'None (request succeeded)'}")
    print()

Query: 'What is your return policy for electronics?'
  Real answer: "Our return policy for electronics allows returns within 30 days of purchase for a full refund, provided the original packaging is included. I don't have any relevant contact details."
  Span: retrieve           status=ok      latency=0.2ms  detail=matched doc: returns
  Span: generate           status=ok      latency=4351.0ms  detail=Our return policy for electronics allows returns within 30 days of purchase for 
  Span: guardrail_check    status=ok      latency=31.1ms  detail=toxicity score 0.001, no PII detected
  Real total latency: 4382.3ms
  Real localized root cause: None (request succeeded)



Query: "What is the CEO's home address?"
  Real answer: "I don't have that information."
  Span: retrieve           status=error   latency=0.0ms  detail=no relevant document found
  Span: generate           status=ok      latency=1138.7ms  detail=I don't have that information.
  Span: guardrail_check    status=ok      latency=24.5ms  detail=toxicity score 0.001, no PII detected
  Real total latency: 1163.3ms
  Real localized root cause: retrieve



Query: 'How do I contact your support team?'
  Real answer: 'You can contact our support team by calling 555-201-4488 or by emailing support@example.com.'
  Span: retrieve           status=ok      latency=0.0ms  detail=matched doc: support
  Span: generate           status=ok      latency=1190.9ms  detail=You can contact our support team by calling 555-201-4488 or by emailing support@
  Span: guardrail_check    status=flagged latency=50.3ms  detail=PII detected: email
  Real total latency: 1241.3ms
  Real localized root cause: guardrail_check



`[REAL]` The 3 real requests produced 3 genuinely different real root-cause outcomes, exactly matching how each was engineered: Request 1 (return policy, a real matching document) succeeded end-to-end -- `localize_root_cause` correctly returned `None`. Request 2 ("CEO's home address," no matching document) real-failed at the `retrieve` span (`status=error`) -- the real `generate` span still ran on empty context and the model gave a real, honest `"I don't have that information."` rather than hallucinating, but `localize_root_cause` still correctly attributes the root cause to the upstream `retrieve` span, not the (correctly-behaving) `generate` span, matching Module 07's own worked-example logic exactly. Request 3 (support contact) succeeded at retrieval and generation, but the real answer legitimately echoed back the support email address from the retrieved document, and the real regex-based PII check correctly flagged it (`status=flagged`, `PII detected: email`) -- `localize_root_cause` correctly attributes this one to `guardrail_check`, the only non-`ok` span. A real, incidental observation: Request 1's `generate` span took `4351.0ms` versus `~1100`-`1200ms` for the other two real calls -- real OpenAI API latency variance, not a designed effect, and a genuine illustration of why per-span (not just aggregate) timing matters for real production diagnosis.

## 5. Capstone Part B: Real Aggregate Evaluation-Set-Versioning Comparison `[SIMULATION]`

`[SIMULATION]` `accuracy` and `diagnose_versioning_failure` are reused verbatim from Module 09. This is a deliberately constructed scenario -- two different real reference-answer sets are authored on purpose to demonstrate the versioning-failure pattern -- kept explicitly separate from Part A above: Part A diagnoses one real request's failure; Part B evaluates aggregate real system behavior across a small evaluation run. Real model outputs as inputs do not make this a real observed production event.

In [6]:
def accuracy(correct, total):
    return correct / total

def diagnose_versioning_failure(acc_v1, acc_v2, model_output_changed):
    if not model_output_changed and acc_v1 != acc_v2:
        return "evaluation-pipeline versioning failure (NOT a real system-quality change)"
    return "real system-quality change (or no change)"

EVAL_QUERIES = [
    "What is your return policy for electronics?",
    "What are your store hours?",
    "How do I contact your support team?",
]
eval_answers = [generate(q, retrieve(q)["text"] if retrieve(q) else "") for q in EVAL_QUERIES]
for q, a in zip(EVAL_QUERIES, eval_answers):
    print(f"Q: {q!r}\n  Real answer: {a!r}")

REFERENCE_V1 = {
    "What is your return policy for electronics?": ["30 days", "refund"],
    "What are your store hours?": ["9am", "8pm"],
    "How do I contact your support team?": ["555-201-4488", "support@example.com"],
}
REFERENCE_V2 = {
    "What is your return policy for electronics?": ["30 days", "refund", "original packaging"],
    "What are your store hours?": ["9am", "8pm", "Sunday", "10am", "6pm"],
    "How do I contact your support team?": ["555-201-4488", "support@example.com"],
}

def score_against_reference(answers, queries, reference):
    correct = 0
    for q, a in zip(queries, answers):
        required = reference[q]
        if all(term.lower() in a.lower() for term in required):
            correct += 1
    return correct, len(queries)

correct_v1, total_v1 = score_against_reference(eval_answers, EVAL_QUERIES, REFERENCE_V1)
correct_v2, total_v2 = score_against_reference(eval_answers, EVAL_QUERIES, REFERENCE_V2)
acc_v1 = accuracy(correct_v1, total_v1)
acc_v2 = accuracy(correct_v2, total_v2)

print(f"\nReal SAME model outputs scored under Reference V1: {correct_v1}/{total_v1} = {acc_v1:.4f}")
print(f"Real SAME model outputs scored under Reference V2: {correct_v2}/{total_v2} = {acc_v2:.4f}")

diagnosis = diagnose_versioning_failure(acc_v1, acc_v2, model_output_changed=False)
print(f"Real diagnosis: {diagnosis}")

Q: 'What is your return policy for electronics?'
  Real answer: "Our return policy for electronics allows returns within 30 days of purchase for a full refund, provided the original packaging is included. I don't have any relevant contact details."
Q: 'What are your store hours?'
  Real answer: "Our store hours are Monday through Saturday, 9am to 8pm, and Sunday 10am to 6pm. I don't have any relevant contact details."
Q: 'How do I contact your support team?'
  Real answer: 'You can contact our support team by calling 555-201-4488 or by emailing support@example.com.'

Real SAME model outputs scored under Reference V1: 3/3 = 1.0000
Real SAME model outputs scored under Reference V2: 3/3 = 1.0000
Real diagnosis: real system-quality change (or no change)


`[SIMULATION]` The real, SAME 3 model outputs scored `3/3=1.0000` under both `REFERENCE_V1` and the stricter `REFERENCE_V2` -- no artifactual accuracy drop was reproduced this time, and `diagnose_versioning_failure` correctly reported `"real system-quality change (or no change)"` rather than falsely flagging a versioning failure that did not actually occur. This honest null result has a clear real explanation: `gpt-4o-mini` at `temperature=0.0`, instructed to answer using the full retrieved context, happened to include every extra term `REFERENCE_V2` required (e.g., `"original packaging"`, `"Sunday"`, `"10am"`, `"6pm"`) as part of its real, complete answers -- so the stricter reference set did not actually penalize anything. This does not undermine Module 09's own point: its hand-verified constructed example (`0.8` vs. `0.6` on the SAME model output) remains the definitive, load-bearing demonstration that reference-version changes can produce a purely artifactual accuracy swing. This notebook's own real attempt to reproduce that pattern at small scale (3 real queries) simply did not happen to surface a case where the model's real answer fell on the wrong side of the stricter reference -- an honest negative result, reported as-is rather than re-engineered post-hoc to force a drop.

## 6. Real Interpretation

`[REAL]` This notebook's real results form a coherent, honest picture, including several genuinely surprising negative findings that turned out to be pedagogically useful in their own right. The real classifier and threshold sweep (Sections 1-2) worked exactly as designed but revealed the test set was too cleanly separable to exercise a genuine precision/recall trade-off -- a real, worth-naming limitation distinct from the classifier being wrong. The real latency experiment (Section 3) directly falsified the naive "parallel is always faster" intuition on this hardware for two checks of very unequal real cost, empirically demonstrating why Module 08's `guardrail_latency` formula has to explicitly state a no-orchestration-overhead assumption -- this notebook's own real measurement shows that assumption does not hold for a per-call `ThreadPoolExecutor`. The capstone's two parts, kept deliberately separate per the signed-off plan, answered two different real questions well: Part A's real per-request trace correctly localized three genuinely different real failure modes (none, upstream retrieval, downstream guardrail flag) using Module 07's own unmodified logic; Part B's `[SIMULATION]` versioning comparison did not reproduce Module 09's constructed artifactual-drop pattern at this small real scale, and that null result is reported honestly rather than forced, with Module 09's own hand-verified example remaining the load-bearing proof that the failure mode is real and possible. Across this whole topic's Track 2, the consistent discipline has been the same: build a real, defensible experiment, run it once, and report exactly what came back -- whether that is a striking finding or, as often here, an honest boring or negative one.